# Arm 2: score the PolyPythias seeds (GPU T4 x2)

Nine independently seeded training runs at each of three sizes, scored on a nested
subsample of the same battery. This is the transport check behind gate G6: it asks
whether the excess `Lambda - 1` measured on DataDecide survives in a population
where nothing varies but the seed.

Session settings: **GPU T4 x2**, internet **on**.

At 500 items per task the battery is about 5,000 items and roughly 20,000 answer
choices per run, and 27 runs fit inside a session at these sizes. Drop
`--n-per-task` to 200 for a pilot; the subsample is nested, so the pilot's items
are a strict subset of the full run's.

In [ ]:
!git clone -q https://github.com/garyzhang1006/seed-noise.git /kaggle/working/seed-noise
!pip -q install -e /kaggle/working/seed-noise

In [ ]:
!pip -q install 'transformers>=4.40' 'datasets>=2.18' 'accelerate>=0.30'

In [ ]:
import torch
print(torch.__version__, torch.cuda.is_available(), torch.cuda.device_count())

## What will be scored

The model ids are the released PolyPythias seeds and the revision is the final
checkpoint. Nothing here trains anything.

In [ ]:
from seednoise.data.polypythias import (
    FINAL_REVISION, SEEDS, SIZES, TASK_SPECS, model_id)

print(FINAL_REVISION, SIZES, SEEDS)
print(sorted(TASK_SPECS) + ["mmlu"])
print(model_id("160m", 3))

## A pilot on one model first

One model, two tasks, 50 items each. It takes a couple of minutes and it fails
loudly if a prompt builder or the tokenizer disagrees with what the scorer
expects, which is much cheaper to find out now.

In [ ]:
from seednoise.data.polypythias import build_items, score_run

items = build_items("arc_easy", n_per_task=50) + build_items("piqa", n_per_task=50)
phen, gain = score_run(items, "70m", 1, max_tokens=30_000)
print(phen.n_items, "items, accuracy", phen.correct.mean().round(4),
      "gain", round(gain, 4))

## The full arm

The token budget caps a padded batch near 30,000 tokens, which fits a T4 at these
sizes with room for the longest HellaSwag continuations. Runs are written as they
finish, so an interrupted session keeps everything already scored.

In [ ]:
!seednoise arm2 \
    --out /kaggle/working/runs-arm2 \
    --n-per-task 500 \
    --max-tokens 30000

## Feed it back into the estimate

Passing `--arm2-runs` adds gate G6, which compares the excess `Lambda - 1` in the
two arms.

In [ ]:
!seednoise analyze \
    --runs /kaggle/input/datadecide-reduced/runs \
    --arm2-runs /kaggle/working/runs-arm2 \
    --out /kaggle/working/results \
    --n-boot 4999

In [ ]:
import pandas as pd
g = pd.read_csv("/kaggle/working/results/tab_gates.csv")
g[g.gate == "G6"].T